# Urban Accessibility Clustering for London, Ontario

This notebook identifies the most underserved census tracts in London, Ontario by measuring accessibility to five essential service categories — hospitals, schools, grocery stores, public transit, and parks — and clusters them using KMeans.

## 1. Project Setup

In [ ]:
# --- Google Colab users: uncomment the two lines below ---
# !pip install -q geopandas osmnx folium scikit-learn seaborn pyogrio
# from google.colab import drive; drive.mount('/content/drive')

import sys
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Project paths
PROJECT_ROOT = Path.cwd().parent          # london-accessibility-gis/
SRC_DIR      = PROJECT_ROOT / "src"
OUTPUT_DIR   = PROJECT_ROOT / "output"

# DATA_DIR: where the raw input files live.
# Adjust this if your data is elsewhere (e.g. Google Drive).
DATA_DIR = PROJECT_ROOT.parent            # ML PROJECT/

# Make src/ importable
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")

In [ ]:
# Core imports
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

# Project modules
from load_data import (
    ensure_output_dirs,
    load_london_census_tracts,
    load_bus_stops,
    load_schools,
    load_hospitals,
    load_parks,
    fetch_grocery_stores_osm,
    load_gtfs_shapes,
)
from feature_engineering import build_accessibility_features
from clustering import (
    CLUSTER_FEATURES,
    prepare_feature_matrix,
    evaluate_kmeans_range,
    choose_optimal_k,
    fit_final_kmeans,
    rank_clusters_by_accessibility,
    apply_cluster_ranking,
    compute_underserved_score,
    summarize_clusters,
)
from visualize import (
    plot_elbow_curve,
    plot_silhouette_scores,
    plot_cluster_heatmap,
    plot_top_underserved,
    create_folium_cluster_map,
)

# Ensure output directory exists
ensure_output_dirs(PROJECT_ROOT)
print("Setup complete.")

## 2. Load Data

In [ ]:
print("=" * 60)
print("Loading census tracts ...")
tracts = load_london_census_tracts(DATA_DIR)

print("\nLoading bus stops ...")
bus_stops = load_bus_stops(DATA_DIR)

print("\nLoading schools ...")
schools = load_schools(DATA_DIR)

print("\nLoading hospitals ...")
hospitals = load_hospitals(DATA_DIR)

print("\nLoading parks ...")
parks = load_parks(DATA_DIR)

print("\nFetching grocery stores from OSM ...")
groceries = fetch_grocery_stores_osm("London, Ontario, Canada")

print("\nLoading GTFS transit shapes ...")
transit_lines = load_gtfs_shapes(DATA_DIR)

print("\n" + "=" * 60)
print("Data loading complete.")

In [ ]:
# Summary table
summary = pd.DataFrame({
    "Layer": ["Census Tracts", "Bus Stops", "Schools", "Hospitals",
              "Parks", "Groceries (OSM)", "Transit Shapes"],
    "Rows": [len(tracts), len(bus_stops), len(schools), len(hospitals),
             len(parks), len(groceries), len(transit_lines)],
    "CRS": [str(tracts.crs), str(bus_stops.crs), str(schools.crs),
            str(hospitals.crs), str(parks.crs), str(groceries.crs),
            str(transit_lines.crs)],
})
summary

## 3. Explore Data

In [ ]:
print("Census tract columns:", list(tracts.columns))
tracts.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tracts.plot(ax=axes[0], edgecolor="black", linewidth=0.5, color="lightblue")
axes[0].set_title("London Census Tracts")

tracts.plot(ax=axes[1], edgecolor="grey", linewidth=0.3, color="#eee")
bus_stops.plot(ax=axes[1], markersize=3, color="red", alpha=0.6)
axes[1].set_title("Bus Stops")

tracts.plot(ax=axes[2], edgecolor="grey", linewidth=0.3, color="#eee")
parks.plot(ax=axes[2], color="green", alpha=0.4)
hospitals.plot(ax=axes[2], markersize=40, color="blue", marker="+")
schools.plot(ax=axes[2], markersize=8, color="orange", alpha=0.7)
axes[2].set_title("Parks, Hospitals, Schools")

for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
gdf = build_accessibility_features(
    tracts, bus_stops, hospitals, schools, parks, groceries, transit_lines
)
gdf.head(3)

In [ ]:
# Check for catastrophic missingness
missing = gdf[CLUSTER_FEATURES].isnull().sum()
print("Missing values per feature:")
print(missing)
print(f"\nTotal tracts: {len(gdf)}")

In [ ]:
gdf[CLUSTER_FEATURES].describe().round(1)

## 5. Clustering Preparation

In [ ]:
print("Clustering features:", CLUSTER_FEATURES)

X_scaled, scaler, X_df = prepare_feature_matrix(gdf, CLUSTER_FEATURES)
print(f"Scaled matrix shape: {X_scaled.shape}")

In [ ]:
print("Evaluating k = 2..8 ...")
ks, inertias, silhouettes = evaluate_kmeans_range(X_scaled, range(2, 9))

## 6. Model Selection

In [ ]:
fig_elbow = plot_elbow_curve(ks, inertias, OUTPUT_DIR)
plt.show()

fig_sil = plot_silhouette_scores(ks, silhouettes, OUTPUT_DIR)
plt.show()

In [ ]:
# Set manual_k=None to use the silhouette-optimal k,
# or set manual_k=<int> to override.
MANUAL_K = None

best_k = choose_optimal_k(silhouettes, ks, manual_k=MANUAL_K)
print(f"\nFinal k = {best_k}")

## 7. Final Clustering

In [ ]:
model, raw_labels = fit_final_kmeans(X_scaled, best_k)
gdf["cluster_raw"] = raw_labels

# Rank clusters: 0 = most accessible, k-1 = least accessible
label_map = rank_clusters_by_accessibility(gdf, CLUSTER_FEATURES, "cluster_raw")
gdf = apply_cluster_ranking(gdf, label_map)

print("Cluster label mapping (raw -> ranked):")
for old, new in sorted(label_map.items(), key=lambda x: x[1]):
    print(f"  raw {old}  ->  ranked {new}")

print(f"\nTracts per ranked cluster:")
print(gdf["cluster"].value_counts().sort_index())

In [ ]:
# Compute continuous underserved score & biggest gap
gdf = compute_underserved_score(
    gdf, CLUSTER_FEATURES, scaler=scaler, X_scaled=X_scaled
)

# Rank tracts
gdf = gdf.sort_values("underserved_score", ascending=False)

print("\nTop 5 most underserved tracts:")
top5_cols = ["CTUID", "cluster", "underserved_score", "biggest_accessibility_gap",
             "dist_nearest_hospital", "dist_nearest_school", "dist_nearest_grocery",
             "num_bus_stops", "num_parks", "transit_coverage"]
top5 = gdf.head(5)[top5_cols].copy()
top5

## 8. Visualization

In [ ]:
fig_hm = plot_cluster_heatmap(gdf, CLUSTER_FEATURES, output_dir=OUTPUT_DIR)
plt.show()

In [ ]:
fig_bar = plot_top_underserved(gdf, n=10, output_dir=OUTPUT_DIR)
plt.show()

In [ ]:
m = create_folium_cluster_map(
    gdf,
    output_path=OUTPUT_DIR / "index.html",
)
m  # displays inline in Jupyter

## 9. Export Outputs

In [ ]:
# 1. Full GeoJSON
geojson_path = OUTPUT_DIR / "london_accessibility_clusters.geojson"
gdf.to_crs("EPSG:4326").to_file(geojson_path, driver="GeoJSON")
print(f"Saved: {geojson_path}")

# 2. Cluster summary CSV
cluster_summary = summarize_clusters(gdf, CLUSTER_FEATURES)
summary_path = OUTPUT_DIR / "cluster_summary.csv"
cluster_summary.to_csv(summary_path)
print(f"Saved: {summary_path}")

# 3. Top 5 underserved CSV
top5_path = OUTPUT_DIR / "top_5_underserved.csv"
top5_export = gdf.head(5)[top5_cols].copy()
top5_export.to_csv(top5_path, index=False)
print(f"Saved: {top5_path}")

print("\nAll outputs written to:", OUTPUT_DIR)

In [ ]:
# Final ranked display
print("=" * 70)
print("TOP 5 MOST UNDERSERVED CENSUS TRACTS")
print("=" * 70)
for i, (_, row) in enumerate(gdf.head(5).iterrows(), 1):
    print(f"\n#{i}  Tract {row['CTUID']}")
    print(f"     Cluster           : {int(row['cluster'])}")
    print(f"     Underserved Score : {row['underserved_score']:.2f}")
    print(f"     Biggest Gap       : {row['biggest_accessibility_gap']}")
    print(f"     Hospital dist     : {row['dist_nearest_hospital']:,.0f} m")
    print(f"     School dist       : {row['dist_nearest_school']:,.0f} m")
    print(f"     Grocery dist      : {row['dist_nearest_grocery']:,.0f} m")
    print(f"     Bus stops         : {int(row['num_bus_stops'])}")
    print(f"     Parks             : {int(row['num_parks'])}")
    print(f"     Transit coverage  : {row['transit_coverage']:,.0f} m")

## 10. Interpretation and Recommendations

### Key Findings

The KMeans clustering partitioned London's census tracts into groups with
distinct accessibility profiles. The **ranked cluster labels** (0 = most
accessible, highest = least accessible) reveal a clear gradient from the
urban core — which benefits from dense transit coverage, numerous bus stops,
and proximity to services — to peripheral tracts where services are sparse.

The **continuous underserved score** goes further than cluster membership
alone, providing a per-tract ranking that accounts for all six accessibility
dimensions simultaneously. The **biggest accessibility gap** column
identifies the single most deficient service for each tract, giving
planners a concrete starting point.

### Priority Investment Recommendations

Based on the analysis, the following investment strategies are suggested
for the most underserved tracts:

1. **Transit expansion** — Tracts with low transit coverage and few bus
   stops should be prioritised for new route extensions or higher-frequency
   service.

2. **Grocery access** — Where the biggest gap is grocery access, consider
   zoning incentives for new grocery/supermarket development or mobile
   market programmes.

3. **Healthcare proximity** — Tracts far from the nearest hospital may
   benefit from satellite clinics or urgent-care centres.

4. **Park and green space** — Low park counts suggest a need for new
   neighbourhood parks or improved access to existing green corridors.

5. **School access** — If school distance is the primary gap, evaluate
   whether additional school-bus routes or safe pedestrian/cycling
   infrastructure could close the deficit.

These recommendations are data-driven starting points. They should be
validated with on-the-ground consultation, population-weighted analysis,
and cost-benefit assessment before implementation.